# Монте-Карло доплеровского охлаждения ¹⁷¹Yb в пинцете
### Этап 6 воспроизведения arXiv:2603.27498 (Song et al.), Appendix D → Fig. 3(a)

**Цель.** Своими руками написать классическое МК-моделирование из Appendix D и получить **Fig. 3(a)**: температуру атома в пинцете как функцию угла $\theta_k$ между пучком охлаждения и осью пинцета для трёх конфигураций (1 ось в симметричной ловушке, 1 ось в анизотропной, 2 оси в анизотропной).

**Логика ноутбука.** Модель собирается из трёх кирпичей, и каждый мы сначала обосновываем, потом пишем:

| § | Что | Откуда в статье | Что пишешь ты |
|---|-----|-----------------|---------------|
| 1 | Иерархия времён: почему движение классическое, а внутреннее состояние — стационарное | — | ничего, только смотришь числа |
| 2 | Внутренние состояния: 4-уровневая система в QuTiP → скорость рассеяния и диаграмма излучения | App. B, C, Eq. (2), (D2) | ✍️ 1: формула (2); ✍️ 2: сэмплер направлений фотонов |
| 3 | Ловушка: потенциал гауссова пучка и сила | Eq. (D1) | ✍️ 3: $U(\vec r)$ и $\nabla U$ |
| 4 | Трение, диффузия, шаг МК; тест в свободном пространстве | Eq. (D2)–(D4) | ✍️ 4: предсказание теории; ✍️ 5: шаг `mc_step` |
| 5 | Атом в пинцете → **Fig. 3(a)** | Sec. III | ✍️ 6: скан и график |
| 6 | Бонус: release–recapture, потери на фотон | Fig. 3(b), 6(a), 7(a) | ✍️ 7, 8 |
| 7 | Список приближений | — | — |

**Как работать.**
* Ячейки с пометкой **✍️ ЗАДАНИЕ** содержат заготовку с `raise NotImplementedError`. Заполни их.
* Сразу после каждого задания идёт **ячейка-проверка** с `assert` и ожидаемыми числами. Пока она не проходит — дальше не иди.
* Застрял — в той же папке лежит `mc_solutions.py` с эталонными решениями: `from mc_solutions import trap_U` и т. п. Сначала попробуй сам.
* Все числа из статьи уже вписаны в ноутбук, искать их не нужно.

**Нужно:** `qutip` (4.7+ или 5.x), `numpy`, `scipy`, `matplotlib`; файл `mc_solutions.py` положи рядом с ноутбуком.

**Время счёта** (numpy, обычный ноутбук): §2–4 — секунды; §5 (Fig. 3a) — около 1.5–3 мин; бонус 6.2 — несколько минут.

## 0. Параметры (всё, что нужно из статьи)

| Величина | Значение | Где в статье |
|---|---|---|
| Переход охлаждения | ¹S₀, F=1/2 → ³P₁, F′=3/2, \|m_F\|=1/2, двухтоновое возбуждение | Sec. II, Fig. 1(b) |
| Естественная ширина | $\Gamma = 2\pi\times 182$ кГц | Fig. 1 |
| Длина волны охлаждения | 555.8 нм | App. A |
| Длина волны пинцета | 759.356 нм (clock-magic) | App. A |
| Перетяжка пинцета | $w_0 = 0.9$ мкм | App. A |
| Масса | 170.936 а.е.м. | — |
| Магнитное поле / ось квантования | 15 Гс вдоль **y** | Sec. III, App. D |
| Ось пинцета | **z** | Sec. III |
| Fig. 3(a): насыщение, отстройка | $s=2$ (полное), $\Delta=-\Gamma$ | подпись Fig. 3 |
| Fig. 3(a): анизотропия | 10 % по перетяжкам | Sec. III |
| Глубина для Fig. 3(a) | **в подписи не указана**; берём 0.2 мК (рабочая глубина статьи) | — |
| Дифф. штарковский сдвиг | $\gamma = 0$ (ловушка «магическая» для перехода охлаждения) | Fig. 3 — по умолчанию |
| Шаг МК | $\delta t = 1$ мкс | App. D |
| Углы пучков в эксперименте | 52° в плоскости xz, 73° в плоскости yz | App. A |

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import qutip as qt
print("qutip", qt.__version__)

# ---------- фундаментальные константы (СИ)
hbar = 1.054571817e-34
kB   = 1.380649e-23
amu  = 1.66053906660e-27

# ---------- атом и свет
m        = 170.9363315 * amu         # 171Yb
Gamma    = 2*np.pi*182e3             # рад/с
lam_cool = 555.80e-9
k        = 2*np.pi/lam_cool

# ---------- пинцет
lam_trap = 759.356e-9
w0       = 0.9e-6
zR       = np.pi*w0**2/lam_trap      # рэлеевская длина

uK = 1e-6                            # удобство: температура в мкК
mK = 1e-3
print(f"zR = {zR*1e6:.2f} мкм")

qutip 5.3.1
zR = 3.35 мкм


## 1. Иерархия времён: почему модель такая

МК в Appendix D — это **полуклассическая модель**: координата и импульс атома классические, а внутреннее (электронное) состояние всё время находится в **стационарном** состоянии, соответствующем текущим $\vec r, \vec v$. Это законно, если выполнены три неравенства. Посчитаем их.

1. **Адиабатическое исключение внутренних степеней свободы.** Внутреннее состояние релаксирует за $\sim 1/\Gamma$. Это время должно быть много меньше периода движения в ловушке $2\pi/\omega$ и времени, за которое заметно меняется доплеровский сдвиг. Тогда в каждый момент атом «видит» стационарную скорость рассеяния $R(\vec r,\vec v)$.
2. **Квазиклассичность движения.** Тепловая энергия много больше кванта колебаний $\hbar\omega$, а импульс отдачи $\hbar k$ много меньше теплового импульса. Тогда движение можно описывать траекторией, а отдачу — малыми случайными толчками (диффузия импульса).
3. **Неразрешённые боковые полосы:** $\omega_r \lesssim \Gamma$. Атом не успевает совершить колебание за время жизни возбуждённого состояния, и охлаждение доплеровское, а не по боковым полосам.

Ячейка ниже ничего не требует от тебя — просто посмотри на числа.

In [3]:
T_typ = 8*uK
U0_typ = 0.2*mK*kB
w_r = np.sqrt(4*U0_typ/(m*w0**2)); w_z = np.sqrt(2*U0_typ/(m*zR**2))
v_th = np.sqrt(kB*T_typ/m)            # 1D тепловая скорость
v_rec = hbar*k/m
Er_h = hbar*k**2/(2*m)/(2*np.pi)
R_typ = Gamma/2 * 2/(1+2+4)           # s=2, Delta=-Gamma

print(f"1/Gamma                  = {1/Gamma*1e6:6.3f} мкс")
print(f"период радиальный / осевой = {2*np.pi/w_r*1e6:6.1f} / {2*np.pi/w_z*1e6:6.1f} мкс")
print(f"omega_r / Gamma          = {w_r/Gamma:6.3f}   (<1: боковые полосы не разрешены)")
print(f"k v_th / Gamma           = {k*v_th/Gamma:6.3f}   (доплеровский сдвиг ~ доли Gamma -> линейная по v сила)")
print(f"kT / (hbar omega_r)      = {kB*T_typ/(hbar*w_r):6.1f}   (>>1: классическое движение)")
print(f"v_rec / v_th             = {v_rec/v_th:6.3f}   (<<1: отдача - малый толчок)")
print(f"E_r/h                    = {Er_h/1e3:6.2f} кГц")
print(f"R(s=2,Delta=-Gamma)      = {R_typ:.3g} 1/с  ->  {R_typ*1e-6:.3f} фотона за шаг 1 мкс")
print(f"фотонов за радиальный период = {R_typ*2*np.pi/w_r:.1f}")

1/Gamma                  =  0.874 мкс
период радиальный / осевой =   28.7 /  151.0 мкс
omega_r / Gamma          =  0.192   (<1: боковые полосы не разрешены)
k v_th / Gamma           =  0.195   (доплеровский сдвиг ~ доли Gamma -> линейная по v сила)
kT / (hbar omega_r)      =    4.8   (>>1: классическое движение)
v_rec / v_th             =  0.213   (<<1: отдача - малый толчок)
E_r/h                    =   3.78 кГц
R(s=2,Delta=-Gamma)      = 1.63e+05 1/с  ->  0.163 фотона за шаг 1 мкс
фотонов за радиальный период = 4.7


**Что из этого следует.**
* $1/\Gamma \approx 0.9$ мкс много меньше радиального периода $\approx 29$ мкс. Значит, внутреннее состояние «успевает» за движением, и можно брать стационарную скорость рассеяния. Это **приближение**: при $\omega_r/\Gamma\approx0.2$ запас всего в 5 раз.
* $k v_{th}\approx 0.2\,\Gamma$, так что силу можно линеаризовать по скорости. Отсюда трение $-\alpha\vec v$ в (D3).
* За шаг $\delta t = 1$ мкс рассеивается в среднем ~0.16 фотона, то есть отдельный шаг — это не много фотонов. Но за время охлаждения (~1 мс) их сотни, и суммарный разброс импульса подчиняется ЦПТ. Поэтому дискретные толчки заменяют гауссовой диффузией с правильной дисперсией (D4). Это ещё одно **приближение**.

## 2. Внутренние состояния: откуда берётся $R_i$ в (D2)

### 2.1 Модель
Базис $\{|g_-\rangle,|g_+\rangle,|e_-\rangle,|e_+\rangle\}$:
* $|g_\pm\rangle = |{}^1S_0,F=\tfrac12,m=\pm\tfrac12\rangle$ — индексы 0, 1;
* $|e_\pm\rangle = |{}^3P_1,F'=\tfrac32,m'=\pm\tfrac12\rangle$ — индексы 2, 3.

**Гамильтониан.** Во вращающейся системе (каждый $|e_{m'}\rangle$ вращается со «своим» тоном) и в RWA ($\hbar=1$, частоты в единицах $\Gamma$):
$$
\hat H = -\Delta\sum_{m'}|e_{m'}\rangle\langle e_{m'}| + \frac12\sum_{m,m'}\Big(\Omega_{m'm}|e_{m'}\rangle\langle g_m| + \text{h.c.}\Big),\qquad
\Omega_{m'm} = \Omega_0\,\langle F m;1 q|F' m'\rangle\,\mathcal E_q,\quad q=m'-m.
$$
Здесь $\langle F m;1q|F'm'\rangle$ — коэффициенты Клебша–Гордана (теорема Вигнера–Эккарта), а $\mathcal E_q$ — сферические компоненты поляризации относительно оси квантования:
$$\mathcal E_0 = E_\parallel,\qquad \mathcal E_{\pm1} = \mp E_\perp/\sqrt2 .$$

**Что выброшено:** встречные члены (RWA); «чужой» тон на каждом $|e_{m'}\rangle$ (он отстроен на $\delta_Z\approx 2\pi\cdot21$ МГц $\approx115\,\Gamma$); уровни $|e_{\pm3/2}\rangle$; зеемановское расщепление основного состояния (~11 кГц).

**Диссипация.** Уравнение Линдблада $\dot\rho = -i[\hat H,\rho] + \sum_q \big(\hat C_q\rho \hat C_q^\dagger - \tfrac12\{\hat C_q^\dagger \hat C_q,\rho\}\big)$, где операторы скачков заданы **по поляризации испущенного фотона** $q$:
$$\hat C_q = \sqrt\Gamma\sum_{m,m'}\langle F m;1 q|F'm'\rangle\,|g_m\rangle\langle e_{m'}| .$$
Задавать их по поляризации (а не по каждому переходу отдельно) важно. Два распада с одинаковым $q$ неотличимы по испущенному фотону, поэтому их амплитуды складываются когерентно. Это создаёт когерентности в основном состоянии, от которых зависит тёмное состояние.

**Скорость рассеяния:** $R = \Gamma\,\mathrm{Tr}(\hat P_e\rho_{ss})$, где $\hat P_e$ — проектор на возбуждённые состояния.

### 2.2 Код (готовый — разбери его)
Поляризация задаётся как в статье: $\vec E = e^{i\phi}E\cos\theta\,\hat e_\perp + E\sin\theta\,\hat e_\parallel$. Интенсивность фиксирована и задаётся параметром $\Omega_0$ (в единицах $\Gamma$).

In [4]:
F_g, F_e = 0.5, 1.5
ms = [-0.5, +0.5]                       # m для g_-, g_+  и  m' для e_-, e_+

def cg(m_g, q, m_e):
    """<F_g m_g; 1 q | F_e m_e>"""
    return qt.clebsch(F_g, 1, F_e, m_g, q, m_e)

def field_spherical(theta, phi=0.0):
    E_par  = np.sin(theta)
    E_perp = np.exp(1j*phi)*np.cos(theta)
    return {0: E_par, +1: -E_perp/np.sqrt(2), -1: +E_perp/np.sqrt(2)}

def coupling_matrix(theta, phi=0.0):
    """V[i,j] = Omega_{m'_i m_j}/Omega_0 ; строки e_-, e_+ ; столбцы g_-, g_+"""
    Eq = field_spherical(theta, phi)
    V = np.zeros((2, 2), complex)
    for i, me in enumerate(ms):
        for j, mg in enumerate(ms):
            q = int(round(me - mg))
            V[i, j] = cg(mg, q, me) * Eq[q]
    return V

def jump_ops():
    """C_q для q = -1, 0, +1 (Gamma = 1)"""
    ops = []
    for q in (-1, 0, +1):
        C = np.zeros((4, 4))
        for j, mg in enumerate(ms):
            for i, me in enumerate(ms):
                if abs(me - mg - q) < 1e-9:
                    C[j, 2 + i] = cg(mg, q, me)      # |g_m><e_m'|
        ops.append(qt.Qobj(C))
    return ops

C_OPS = jump_ops()
P_E = qt.ket2dm(qt.basis(4, 2)) + qt.ket2dm(qt.basis(4, 3))

def build_H(Omega0, Delta, theta, phi=0.0):
    """Delta, Omega0 — в единицах Gamma."""
    V = Omega0*coupling_matrix(theta, phi)
    H = np.zeros((4, 4), complex)
    H[2, 2] = H[3, 3] = -Delta
    H[2:, :2] = V/2
    H[:2, 2:] = V.conj().T/2
    return qt.Qobj(H), V

def rate_qutip(Omega0, Delta, theta, phi=0.0):
    """R/Gamma из стационарного решения уравнения Линдблада."""
    H, _ = build_H(Omega0, Delta, theta, phi)
    rho = qt.steadystate(H, C_OPS)
    return qt.expect(P_E, rho)

def s_pm(Omega0, theta, phi=0.0):
    """Моррис-Шор: Omega_± — сингулярные числа V  ->  s_± = 2 Omega_±^2 (Gamma = 1)."""
    sv = np.linalg.svd(Omega0*coupling_matrix(theta, phi), compute_uv=False)
    return 2*sv**2

print("КГ^2: pi =", cg(0.5, 0, 0.5)**2, ", sigma =", cg(-0.5, 1, 0.5)**2)
print("V(theta=30°, phi=0) / Omega_0 =\n", np.round(coupling_matrix(np.deg2rad(30)), 4))
print("s_± при Omega0=2, theta=30°, phi=0    :", s_pm(2, np.deg2rad(30)))
print("s_± при Omega0=2, theta=26.57°, phi=pi/2:", s_pm(2, np.arctan(0.5), np.pi/2), " <- тёмное состояние")

КГ^2: pi = 0.6666666666666666 , sigma = 0.3333333333333333
V(theta=30°, phi=0) / Omega_0 =
 [[ 0.4082+0.j  0.3536+0.j]
 [-0.3536+0.j  0.4082+0.j]]
s_± при Omega0=2, theta=30°, phi=0    : [2.33333333 2.33333333]
s_± при Omega0=2, theta=26.57°, phi=pi/2: [4.26666667e+00 1.63253111e-32]  <- тёмное состояние


Обрати внимание на вывод.
* При $\phi=0$ (линейная поляризация) оба сингулярных числа равны: $V$ пропорциональна унитарной матрице, и система — это **две одинаковые** двухуровневые системы. Тёмного состояния нет.
* При $\phi=\pi/2$ и $\tan\theta=1/2$ одно сингулярное число равно нулю: это тёмное состояние Fig. 2(c).

### ✍️ ЗАДАНИЕ 1. Формула (2)
Для двух независимых двухуровневых систем $|g_\pm\rangle-|e_\pm\rangle$ скорости рассеяния равны $R_\pm = \frac\Gamma2\frac{s_\pm}{1+s_\pm+(2\Delta/\Gamma)^2}$. Населённости основных состояний обратно пропорциональны скоростям «выкачки» (App. C). Отсюда
$$R_{\rm tot} = \frac{2R_+R_-}{R_++R_-} = \frac\Gamma2\,\frac{s_{\rm tot}}{1+s_{\rm tot}+(2\Delta/\Gamma)^2},\qquad s_{\rm tot}=\frac{2s_+s_-}{s_++s_-}.$$
(Проверь алгебру второго равенства на бумаге: подставь $R_\pm$ и приведи к общему знаменателю.)

Напиши `R_formula(sp, sm, Delta)`, которая возвращает $R_{\rm tot}/\Gamma$, где `Delta` задано в единицах $\Gamma$. Если $s_++s_-=0$, верни 0.

In [5]:
# ✍️ ЗАДАНИЕ
def R_formula(sp, sm, Delta):
    """R_tot/Gamma по Eq. (2). sp, sm — s_+ и s_-; Delta — в единицах Gamma."""
    # ✍️ ВАШ КОД
    raise NotImplementedError

In [6]:
# ПРОВЕРКА 1: QuTiP (полное 4-уровневое уравнение Линдблада) против Eq. (2)
rng_chk = np.random.default_rng(0)
errs = []
for _ in range(30):
    Om0, D = rng_chk.uniform(0.3, 3), rng_chk.uniform(-2, 0.5)
    th, ph = rng_chk.uniform(0, np.pi/2), rng_chk.choice([0, np.pi/2, rng_chk.uniform(0, np.pi)])
    sp, sm = s_pm(Om0, th, ph)
    a, b = rate_qutip(Om0, D, th, ph), R_formula(sp, sm, D)
    errs.append(abs(a - b)/max(a, 1e-12))
print("макс. отн. ошибка:", max(errs))
assert max(errs) < 1e-6, "Eq. (2) не совпала с QuTiP — проверь R_formula"
print("✅ ок")

# Аналог Fig. 2(c,d): Omega0 подобран так, что s = 2 при theta = 0; Delta = -0.92 Gamma
th = np.linspace(0, np.pi/2, 46)
Om0 = np.sqrt(2/(2*abs(coupling_matrix(0)[0, 1])**2))
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for a_, ph in zip(ax, [np.pi/2, 0]):
    Rq = [rate_qutip(Om0, -0.92, t, ph) for t in th]
    Rf = [R_formula(*s_pm(Om0, t, ph), -0.92) for t in th]
    det = 0.0224*np.array(Rq)*Gamma/1e3          # эффективность детектирования 2.24 % (App. A)
    a_.plot(np.rad2deg(th), det, 'o', ms=3, label='QuTiP')
    a_.plot(np.rad2deg(th), 0.0224*np.array(Rf)*Gamma/1e3, '-', label='Eq. (2)')
    a_.set_title(f"phi = {ph:.2f}"); a_.set_xlabel("theta (deg)")
ax[0].set_ylabel("детект. фотоны, 10³/с"); ax[0].legend(); plt.tight_layout(); plt.show()

NotImplementedError: 

**Ожидаемый результат:** совпадение до ~1e-10, причём **и при $\phi=\pi/2$**. Для этой модели Eq. (2) точна. Графики повторяют штриховые линии Fig. 2(c,d): провал до нуля около 26.6° при $\phi=\pi/2$; от ≈4·10³ до ≈8·10³ с⁻¹ при $\phi=0$.

**Вывод для МК.** В работе используется линейная поляризация ($\phi=0$). Тогда $s_+=s_-=s$, и атом рассеивает как **двухуровневый** атом с насыщением $s$. Именно поэтому в (D2) стоит простая двухуровневая формула. Для нескольких пучков сумма $\sum_i s_i$ в знаменателе означает, что интенсивности пучков **складываются некогерентно**. Это приближение: интерференция между пучками выброшена (о её роли — Fig. 3(b) и Sec. III).

### 2.3 Диаграмма излучения — куда летит испущенный фотон
В (D2)–(D4) случайный толчок $\hat e_p$ выбирается «с учётом дипольной диаграммы относительно оси квантования (y)». Выведем эту диаграмму.

Фотон с поляризацией $q$ испускается диполем $\vec d\propto\hat e_q$. Интенсивность в направлении $\hat n$ пропорциональна $1-|\hat n\cdot\hat e_q|^2$ (поперечность излучения). Пусть $\vartheta$ — угол между $\hat n$ и осью квантования. Тогда:
* $\pi$ ($q=0$, $\hat e_0=\hat y$): $\;\frac{dP}{d\Omega} = \frac{3}{8\pi}\sin^2\vartheta$;
* $\sigma^\pm$ ($\hat e_{\pm1}=\mp(\hat x\pm i\hat z)/\sqrt2$ в осях с полярной осью y): $\;1-\tfrac12\sin^2\vartheta = \tfrac12(1+\cos^2\vartheta)$, после нормировки $\frac{3}{16\pi}(1+\cos^2\vartheta)$.

Веса $w_q$ — доли фотонов каждой поляризации — это $\langle\hat C_q^\dagger\hat C_q\rangle/\sum_q\langle\hat C_q^\dagger\hat C_q\rangle$ в стационарном состоянии. Посчитаем их в QuTiP:

In [ ]:
H, _ = build_H(np.sqrt(2/(2*abs(coupling_matrix(0)[0, 1])**2)), -1.0, np.deg2rad(40))
rho_ss = qt.steadystate(H, C_OPS)
rates_q = np.array([qt.expect(C.dag()*C, rho_ss) for C in C_OPS])
w = rates_q/rates_q.sum()
print("доли фотонов  sigma-, pi, sigma+ :", np.round(w, 6))

Получилось $w_\pi = 2/3$, $w_{\sigma^+}=w_{\sigma^-}=1/6$, причём **при любой поляризации и отстройке** (проверь, меняя аргументы). Причина: каждое из $|e_{\pm1/2}\rangle$ распадается по $\pi$ с вероятностью $|{\rm КГ}|^2=2/3$ и по $\sigma$ с вероятностью $1/3$.

Итоговая диаграмма:
$$
\frac{dP}{d\Omega} = \frac23\cdot\frac{3}{8\pi}\sin^2\vartheta+\frac13\cdot\frac{3}{16\pi}(1+\cos^2\vartheta)
=\frac{1}{16\pi}\big(4\sin^2\vartheta+1+\cos^2\vartheta\big)=\frac{5-3\cos^2\vartheta}{16\pi}.
$$
Проверка нормировки: $\int d\Omega = 2\pi\int_{-1}^1\frac{5-3u^2}{16\pi}du=\frac{2\pi\cdot 8}{16\pi}=1$.

Плотность по $u=\cos\vartheta$: $p(u)=(5-3u^2)/8$ на $[-1,1]$. Азимут $\varphi$ равномерен. Полезные моменты: $\langle u^2\rangle = 4/15$, $\langle n_x^2\rangle=\langle n_z^2\rangle = 11/30$. Вдоль оси квантования улетает меньше фотонов.

### ✍️ ЗАДАНИЕ 2. Сэмплер направлений
Напиши `sample_emission_dirs(n, rng)`, которая возвращает массив `(n, 3)` единичных векторов с плотностью $(5-3\cos^2\vartheta)/16\pi$, где **ось квантования — y**. Компоненты: $\vec n = (\sin\vartheta\cos\varphi,\; \cos\vartheta,\; \sin\vartheta\sin\varphi)$.

*Подсказка (метод отбраковки):* бери $u\sim U[-1,1]$ и принимай с вероятностью $(5-3u^2)/5$ (максимум плотности равен 5). Делай это векторно, пока не наберёшь $n$ штук.

In [ ]:
# ✍️ ЗАДАНИЕ
def sample_emission_dirs(n, rng):
    """(n,3) единичных векторов с диаграммой (5-3cos^2)/(16 pi), ось квантования = y."""
    # ✍️ ВАШ КОД
    raise NotImplementedError

In [ ]:
# ПРОВЕРКА 2
e = sample_emission_dirs(400_000, np.random.default_rng(1))
assert e.shape == (400_000, 3)
assert np.allclose(np.linalg.norm(e, axis=1), 1)
m2 = (e**2).mean(axis=0)
print("<n_x^2>, <n_y^2>, <n_z^2> =", np.round(m2, 4), "  ожидание: 0.3667, 0.2667, 0.3667")
assert abs(m2[1] - 4/15) < 3e-3 and abs(m2[0] - 11/30) < 3e-3
print("✅ ок")
uu = np.linspace(-1, 1, 200)
plt.figure(figsize=(4.5, 3))
plt.hist(e[:, 1], bins=60, density=True, alpha=.5, label="сэмплы")
plt.plot(uu, (5 - 3*uu**2)/8, label="(5-3u²)/8"); plt.xlabel("u = cos ϑ (относительно y)")
plt.legend(); plt.tight_layout(); plt.show()

### 2.4 (необязательно) Квантовые скачки: насколько оправдана диффузия
`qt.mcsolve` моделирует отдельные траектории с испусканием фотонов. По ним видно, сколько фотонов приходится на шаг МК $\delta t = 1$ мкс и что фотоны вылетают поодиночке. Ячейка считает ~10–30 с.

In [ ]:
Om0 = np.sqrt(2/(2*abs(coupling_matrix(0)[0, 1])**2))      # s = 2 при theta = 0
H, _ = build_H(Om0, -1.0, 0.0)                              # s=2, Delta=-Gamma
psi0 = qt.basis(4, 0)
tlist = np.linspace(0, 200, 201)                            # в единицах 1/Gamma (~175 мкс)
res = qt.mcsolve(H, psi0, tlist, C_OPS, ntraj=100)
times = np.concatenate([np.asarray(t) for t in res.col_times])
R_sim = len(times)/(100*tlist[-1])
print(f"скорость из траекторий: {R_sim:.4f} Gamma   стационарная: {rate_qutip(Om0, -1.0, 0.0):.4f} Gamma")
dt_G = 1e-6*Gamma                                           # 1 мкс в единицах 1/Gamma
counts = np.concatenate([np.histogram(np.asarray(t), bins=np.arange(20, 200, dt_G))[0] for t in res.col_times])
vals, cnt = np.unique(counts, return_counts=True)
print("фотонов за 1 мкс: значение -> доля", dict(zip(vals.tolist(), np.round(cnt/cnt.sum(), 3).tolist())))

**Ожидаемо:** около 0.16 фотона за 1 мкс; в ~84 % шагов фотонов нет. Значит, отдельный шаг — не гауссов процесс. Но интегрально (сотни фотонов за время охлаждения) работает только дисперсия, и гауссова диффузия с правильным $D_p$ даёт ту же температуру. Альтернатива — честно разыгрывать дискретные фотоны; это хорошее упражнение «на потом».

## 3. Ловушка

### 3.1 Потенциал
Дипольный потенциал пропорционален интенсивности: $U(\vec r) = -\frac{\mathrm{Re}\,\alpha_{\rm pol}}{2\epsilon_0 c}I(\vec r)$. Для гауссова пучка вдоль $z$
$$I \propto \frac{w_0^2}{w(z)^2}\exp\!\Big(-\frac{2\rho^2}{w(z)^2}\Big),\qquad w(z)^2=w_0^2\big(1+(z/z_R)^2\big),\quad z_R=\frac{\pi w_0^2}{\lambda_{\rm trap}} .$$
Отсюда (D1). Анизотропию учитываем, разрешая разные перетяжки по $x$ и $y$. Это упрощение: $z_R$ оставляем от $w_0$. Обозначим $q\equiv 1+(z/z_R)^2$ и $A\equiv 2\left(x^2/w_x^2+y^2/w_y^2\right)$:
$$U(\vec r) = -\frac{U_0}{q}\,e^{-A/q}.$$

### 3.2 Гармоническое приближение
Разложение около нуля: $U\approx -U_0 + \frac{2U_0}{w_x^2}x^2 + \frac{2U_0}{w_y^2}y^2 + \frac{U_0}{z_R^2}z^2$. Это следует из $e^{-A}\approx1-A$ и $1/q\approx 1-z^2/z_R^2$. Сравнивая с $\tfrac12 m\omega^2 x^2$:
$$\omega_{x}=\sqrt{\frac{4U_0}{m w_x^2}},\qquad \omega_z=\sqrt{\frac{2U_0}{m z_R^2}} .$$

### 3.3 Градиент (выведи сам, потом сверься)
Дифференцируй $U=-U_0 q^{-1}e^{-A/q}$ по $x$ (через $A$) и по $z$ (через $q$, $dq/dz=2z/z_R^2$).

<details><summary>Ответ (раскрой после вывода)</summary>

$$\partial_x U = U\cdot\Big(-\frac{4x}{w_x^2 q}\Big),\quad \partial_y U = U\cdot\Big(-\frac{4y}{w_y^2 q}\Big),\quad \partial_z U = U\cdot\frac{A/q-1}{q}\cdot\frac{2z}{z_R^2}.$$
</details>

### ✍️ ЗАДАНИЕ 3
Напиши `trap_U(r, U0, wx, wy)` и `trap_gradU(r, U0, wx, wy)`. Вход `r` — массив `(N,3)`. Выход: `U` формы `(N,)` и пара `(U, gradU)`, где `gradU` имеет форму `(N,3)`.

In [ ]:
# ✍️ ЗАДАНИЕ
def trap_U(r, U0, wx=w0, wy=w0):
    # ✍️ ВАШ КОД
    raise NotImplementedError

def trap_gradU(r, U0, wx=w0, wy=w0):
    """возвращает (U, gradU)"""
    # ✍️ ВАШ КОД
    raise NotImplementedError

In [ ]:
def trap_freqs(U0, wx=w0, wy=w0):
    """(f_x, f_y, f_z) в Гц — гармоническое приближение"""
    return (np.sqrt(4*U0/(m*wx**2))/(2*np.pi), np.sqrt(4*U0/(m*wy**2))/(2*np.pi),
            np.sqrt(2*U0/(m*zR**2))/(2*np.pi))

# ПРОВЕРКА 3
U0 = 0.2*mK*kB
wy_a = 1.1*w0
assert np.isclose(trap_U(np.zeros((1, 3)), U0)[0], -U0)
rr = np.random.default_rng(2).normal(size=(50, 3))*np.array([0.4e-6, 0.4e-6, 2e-6])
_, g = trap_gradU(rr, U0, w0, wy_a)
h = 1e-11; g_num = np.zeros_like(rr)
for i in range(3):
    d = np.zeros(3); d[i] = h
    g_num[:, i] = (trap_U(rr + d, U0, w0, wy_a) - trap_U(rr - d, U0, w0, wy_a))/(2*h)
err = np.max(np.abs(g - g_num))/np.max(np.abs(g))
print("отн. ошибка градиента:", err); assert err < 1e-5
# частоты из второй производной численно
h = 1e-9; f_num = []
for i in range(3):
    d = np.zeros((1, 3)); d[0, i] = h
    k2 = (trap_U(d, U0, w0, wy_a)[0] - 2*trap_U(np.zeros((1, 3)), U0, w0, wy_a)[0] + trap_U(-d, U0, w0, wy_a)[0])/h**2
    f_num.append(np.sqrt(k2/m)/(2*np.pi))
print("частоты (числ.)  кГц:", np.round(np.array(f_num)/1e3, 2))
print("частоты (форм.)  кГц:", np.round(np.array(trap_freqs(U0, w0, wy_a))/1e3, 2))
print("симметричная 0.2 мК:", np.round(np.array(trap_freqs(U0))/1e3, 2), " ожидание: 34.88, 34.88, 6.62")
print("симметричная 0.14 мК:", np.round(np.array(trap_freqs(0.14*mK*kB))/1e3, 2), " (линии Fig. 7(b): 29.2 и 5.5)")
print("✅ ок")

x = np.linspace(-2, 2, 200)*1e-6; z = np.linspace(-8, 8, 200)*1e-6
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for lab, wx_, wy_ in [("x", w0, wy_a), ("y", w0, wy_a)]:
    r_ = np.zeros((200, 3)); r_[:, 0 if lab == "x" else 1] = x
    ax[0].plot(x*1e6, trap_U(r_, U0, wx_, wy_)/kB/mK, label=lab)
r_ = np.zeros((200, 3)); r_[:, 2] = z
ax[1].plot(z*1e6, trap_U(r_, U0)/kB/mK)
ax[0].set_xlabel("мкм"); ax[1].set_xlabel("z, мкм"); ax[0].set_ylabel("U, мК"); ax[0].legend()
plt.tight_layout(); plt.show()

**Физика.** Осевая частота в ~5 раз меньше радиальной, ловушка сильно **ангармонична** (тепловая энергия ~8 мкК при глубине 200 мкК). Ангармонизм перемешивает энергию между осями: частота зависит от амплитуды, и моды связаны через $A/q$. Именно поэтому даже **один** наклонный пучок охлаждает все три степени свободы. В анизотропной ловушке $x$ и $y$ не вырождены, перемешивание между ними слабее, и одна ось охлаждает хуже. Это синяя кривая Fig. 3(a).

## 4. Сила, трение, диффузия: модель Appendix D

### 4.1 Трение от пары встречных пучков
Один пучок с волновым вектором $\hbar k\hat k$ и насыщением $s_b$ давит с силой $\hbar k\hat k\,\frac\Gamma2\frac{s_b}{1+S+(\delta - 2k\hat k\cdot\vec v/\Gamma)^2}$, где $\delta=2\Delta/\Gamma$. Атом видит отстройку, сдвинутую эффектом Доплера, а $S$ — полное насыщение всеми пучками (некогерентная сумма). Сложим силы от $+\hat k$ и $-\hat k$ и разложим до первого порядка по $\epsilon=2k\hat k\cdot\vec v/\Gamma$. Используем $f(\delta-\epsilon)-f(\delta+\epsilon)\approx -2\epsilon f'(\delta)$ при $f(x)=\frac1{1+S+x^2}$:
$$\vec F = \hbar k\hat k\,\frac\Gamma2\,s_b\cdot\frac{4\epsilon\,\delta}{(1+S+\delta^2)^2}
= -\underbrace{\Big(-4\hbar k^2 s_b\frac{\delta}{(1+S+\delta^2)^2}\Big)}_{\alpha_1}(\hat k\cdot\vec v)\,\hat k .$$
В статье $s_i$ — насыщение **пары**, то есть $s_b = s_i/2$. Это и есть «/2» в (D3). При $\delta<0$ имеем $\alpha_1>0$: трение.

### 4.2 Обобщение статьи на 3D
Вместо векторной силы $-\sum_i\alpha_{1,i}(\hat k_i\cdot\vec v)\hat k_i$ в (D3) стоит **скалярное** трение вдоль $\vec v$:
$$\alpha = -4\hbar k^2\,\frac{\sum_i s_i(\hat k_i\cdot\hat v)^2}{2}\,\frac{\delta'}{\big(1+\sum_i s_i+\delta'^2\big)^2},\qquad \vec F_{\rm fr}=-\alpha\vec v .$$
Обе силы рассеивают **одинаковую мощность**: $\vec F\cdot\vec v = -\alpha_1\sum_i(\hat k_i\cdot\vec v)^2$. Различаются они только направлением (у статьи компонента поперёк $\vec v$ отброшена). Это **приближение**, и его можно проверить, заменив силу на векторную.

### 4.3 Диффузия
Каждый акт рассеяния — это поглощение (толчок $\hbar k$ вдоль $\pm\hat k_i$) плюс испускание (толчок $\hbar k$ в случайном направлении). Толчки некоррелированы, поэтому $\langle|\Delta\vec p|^2\rangle = 2\hbar^2k^2$ на акт. При скорости $\sum_i R_i$:
$$\frac{d\langle|\vec p|^2\rangle}{dt}\Big|_{\rm heat} = 2\hbar^2k^2\sum_iR_i\equiv 2D_p\quad\Rightarrow\quad D_p=\hbar^2k^2\sum_i R_i\quad\text{(D4)}.$$
В статье весь нагрев за шаг даётся **одним** толчком $\hat e_p\sqrt{2D_p\delta t}$ с направлением из дипольной диаграммы (§2.3). Полная энергия нагрева верная. Но распределение нагрева по осям приближённое: поглощение на самом деле идёт вдоль пучка, а не по диаграмме.

### 4.4 Неоднородный световой сдвиг
Если поляризуемость возбуждённого состояния отличается от основного, то $U_e=(1+\gamma)U$. Тогда:
* резонанс сдвигается в зависимости от положения: $\Delta' = \Delta + \gamma\,(U_0+U(\vec r))/\hbar$. На дне ловушки $\Delta'=\Delta$, то есть $\Delta$ отсчитана от резонанса **в ловушке**;
* атом проводит долю времени $\rho_{ee}=\sum_iR_i/\Gamma$ в возбуждённом состоянии и в среднем чувствует $U'=(1-\rho_{ee})U+\rho_{ee}(1+\gamma)U = U\,(1+\gamma\rho_{ee})$.

Сила равна $-\nabla U'\approx-(1+\gamma\rho_{ee})\nabla U$; градиентом $\rho_{ee}$ пренебрегаем. Для Fig. 3(a) берём $\gamma=0$, но в коде параметр оставляем.

### 4.5 Шаг МК (как в статье)
$$\vec v\to\vec v+\big(-\nabla U'-\alpha\vec v\big)\frac{\delta t}{m}+\hat e_p\frac{\sqrt{2D_p\delta t}}{m},\qquad \vec r\to\vec r+\vec v\,\delta t\ \ (\text{новой } \vec v),$$
Это полунеявный (симплектический) Эйлер: устойчив при $\omega\delta t<2$, у нас $\omega_r\delta t\approx0.22$. Атом **потерян**, если $E=\tfrac12mv^2+U(\vec r)\ge0$.

### 4.6 Чего ждать в свободном пространстве
Пусть перемешивание идеальное (распределение $\vec v$ изотропно). Тогда $\langle(\hat k\cdot\hat v)^2\rangle = 1/3$ на каждую ось, и баланс энергии
$$\frac{d\langle|\vec p|^2\rangle}{dt} = 2D_p - \frac{2}{m}\,\frac{\alpha_0}{3}\langle|\vec p|^2\rangle,\qquad \alpha_0=-2\hbar k^2 s\frac{\delta}{(1+s+\delta^2)^2}\ \ (s=\textstyle\sum_i s_i)$$
даёт $\langle|\vec p|^2\rangle = 3mD_p/\alpha_0$, то есть $k_BT = \langle|\vec p|^2\rangle/3m = D_p/\alpha_0$:
$$\boxed{k_BT=\frac{\hbar\Gamma}{4}\,\frac{1+s+\delta^2}{|\delta|}}\qquad\Rightarrow\qquad \delta_{\rm opt}=-\sqrt{1+s},\quad T_{\min}=\frac{\hbar\Gamma}{2k_B}\sqrt{1+s}\approx 4.37\ \text{мкК}\cdot\sqrt{1+s}.$$
Это стандартный доплеровский результат (Lett et al., [36]). Тот же ответ даёт геометрия из трёх ортогональных пар пучков.

### ✍️ ЗАДАНИЕ 4. Предсказание теории
Напиши `T_doppler_model(s, Delta)`, возвращающую $T$ в кельвинах (`Delta` в рад/с).

In [ ]:
# ✍️ ЗАДАНИЕ
def T_doppler_model(s, Delta):
    # ✍️ ВАШ КОД
    raise NotImplementedError

In [ ]:
# ПРОВЕРКА 4
print(f"s=2, Delta=-Gamma : T = {T_doppler_model(2, -Gamma)/uK:.3f} мкК   (ожидание 7.643)")
assert abs(T_doppler_model(2, -Gamma)/uK - 7.643) < 0.01
print("минимумы для s = 0.1, 1, 2, 3:",
      [round(T_doppler_model(s, -np.sqrt(1+s)*Gamma/2)/uK, 2) for s in (0.1, 1, 2, 3)],
      " (МК статьи в пинцете: 4.9, 7.1, 8.8, 10.3)")
print("✅ ок")

### ✍️ ЗАДАНИЕ 5. Шаг Монте-Карло — сердце моделирования
Реализуй `mc_step`. Работай **векторно** по ансамблю из $N$ атомов, без циклов по атомам.

Входы:
* `r, v` — `(N,3)`, обновляются **на месте**;
* `alive` — `(N,)` bool, обновляется на месте; потерянные атомы больше не двигаются;
* `s_vec` — `(n_axes,)` насыщения пар (у выключенной оси 0);
* `K` — `(n_axes,3)` единичные $\hat k_i$;
* `Delta` (рад/с), `gamma`, `U0` (Дж), `wx, wy`, `dt`, `rng`;
* `dirs_table` — заранее насэмпленные направления (сэмплировать на каждом шаге дорого). Бери случайные строки через `rng.integers`.

Возвращает `Rsum` формы `(N,)` — полную скорость рассеяния $\sum_iR_i$ (0 для потерянных). Она нужна для счёта фотонов.

Шаги (номера формул из статьи):
1. `U, gU = trap_gradU(...)`.
2. $\Delta'$ и $\delta'=2\Delta'/\Gamma$ (§4.4).
3. Знаменатель $1+\sum s_i+\delta'^2$ и $\sum_iR_i$ (D2).
4. Сила $-\nabla U'=-(1+\gamma\rho_{ee})\nabla U$.
5. $(\hat k_i\cdot\hat v)^2$ для всех осей (подсказка: `(v @ K.T)**2 / |v|^2`, к $|v|^2$ добавь `1e-300` против деления на 0) и $\alpha$ (D3).
6. $D_p$ (D4) и направления $\hat e_p$.
7. Обнови `v`, затем `r` — только для `alive` (`np.where`).
8. Если `U0 > 0`: пометь потерянными атомы с $E\ge0$. В свободном пространстве (`U0 = 0`) никого не теряем.

In [ ]:
# ✍️ ЗАДАНИЕ
def mc_step(r, v, alive, s_vec, K, Delta, gamma, U0, wx, wy, dt, rng, dirs_table):
    N = r.shape[0]
    # ✍️ ВАШ КОД (шаги 1–8)
    raise NotImplementedError

### 4.7 Обвязка (готовая)
`run_mc` раздаёт начальные условия (максвелловское распределение в гармоническом приближении), крутит `mc_step` и собирает статистику. Температура считается как $T=m\langle|\vec v|^2\rangle/3k_B$ по **выжившим** атомам, усреднённая по последним `t_meas` секундам.

Пучок под углом $\theta_k$ к оси $z$: в плоскости xz $\hat k=(\sin\theta_k,0,\cos\theta_k)$, в плоскости yz $\hat k=(0,\sin\theta_k,\cos\theta_k)$.

Полная интенсивность фиксирована (как в эксперименте). При одновременной работе осей она делится поровну: $s_i = s/n_{\rm axes}$. В режиме чередования (`alt_freq`) вся $s$ отдаётся активной оси.

In [ ]:
def khat(theta_deg, plane):
    t = np.deg2rad(theta_deg)
    return np.array([np.sin(t), 0, np.cos(t)]) if plane == 'xz' else np.array([0, np.sin(t), np.cos(t)])

def init_thermal(N, T0, U0, wx, wy, rng):
    fx, fy, fz = trap_freqs(U0, wx, wy)
    sig_r = np.sqrt(kB*T0/m)/(2*np.pi*np.array([fx, fy, fz]))
    return rng.normal(size=(N, 3))*sig_r, rng.normal(size=(N, 3))*np.sqrt(kB*T0/m)

def run_mc(N=400, t_total=10e-3, t_meas=5e-3, s=2.0, Delta=-Gamma, gamma=0.0, U0=0.2*mK*kB,
           aniso=0.0, axes=(('xz', 50.0),), alt_freq=None, dt=1e-6, T0=20*uK, seed=0, free_space=False):
    rng = np.random.default_rng(seed)
    dirs_table = sample_emission_dirs(200_000, rng)
    wx, wy = w0, w0*(1 + aniso)
    K = np.array([khat(th, pl) for pl, th in axes]); nax = len(axes)
    if free_space:
        U0_ = 0.0; r = np.zeros((N, 3)); v = rng.normal(size=(N, 3))*np.sqrt(kB*T0/m)
    else:
        U0_ = U0; r, v = init_thermal(N, T0, U0, wx, wy, rng)
    alive = np.ones(N, bool); photons = np.zeros(N)
    nsteps = int(round(t_total/dt)); n0 = nsteps - int(round(t_meas/dt))
    T_t, ret_t, nph_t = np.empty(nsteps), np.empty(nsteps), np.empty(nsteps)
    acc = cnt = 0
    for n in range(nsteps):
        if alt_freq is None:
            s_vec = np.full(nax, s/nax)
        else:
            s_vec = np.zeros(nax); s_vec[int(2*alt_freq*n*dt) % nax] = s
        photons += mc_step(r, v, alive, s_vec, K, Delta, gamma, U0_, wx, wy, dt, rng, dirs_table)*dt
        if alive.any():
            ke = m*np.einsum('ij,ij->i', v[alive], v[alive]).mean()/(3*kB); nph_t[n] = photons[alive].mean()
        else:
            ke = nph_t[n] = np.nan
        T_t[n] = ke; ret_t[n] = alive.mean()
        if n >= n0:
            acc += ke; cnt += 1
    return dict(T=acc/cnt/uK, retention=alive.mean(), photons=photons, alive=alive, t=np.arange(nsteps)*dt,
                T_t=T_t/uK, ret_t=ret_t, nph_t=nph_t, r=r, v=v, wx=wx, wy=wy)

In [ ]:
# ПРОВЕРКА 5а: без света в ловушке энергия сохраняется (проверяет силу и интегратор)
rng_t = np.random.default_rng(5)
r_, v_ = init_thermal(200, 10*uK, 0.2*mK*kB, w0, w0, rng_t)
al = np.ones(200, bool); K1 = np.array([khat(50, 'xz')]); tab = sample_emission_dirs(1000, rng_t)
E = lambda: (0.5*m*np.einsum('ij,ij->i', v_, v_) + trap_U(r_, 0.2*mK*kB)).mean()
Es = []
for n in range(3000):
    mc_step(r_, v_, al, np.array([0.0]), K1, -Gamma, 0.0, 0.2*mK*kB, w0, w0, 1e-6, rng_t, tab); Es.append(E())
Es = np.array(Es)/kB/uK
drift = abs(Es[-300:].mean() - Es[:300].mean())/abs(Es.mean())
print(f"<E> = {Es.mean():.2f} мкК·kB, дрейф за 3 мс: {drift:.2e}"); assert drift < 2e-3 and al.all()

# ПРОВЕРКА 5б: свободное пространство, 3 ортогональные пары, s=2, Delta=-Gamma  ->  ~7.6 мкК
res = run_mc(N=400, t_total=4e-3, t_meas=2e-3, free_space=True, T0=10*uK,
             axes=(('xz', 0), ('xz', 90), ('yz', 90)))
print(f"МК: T = {res['T']:.2f} мкК     теория: {T_doppler_model(2, -Gamma)/uK:.2f} мкК")
assert abs(res['T'] - 7.64) < 0.6, "свободное пространство не сошлось с теорией §4.6"
print("✅ ок")

**Ожидаемо:** ~7.6–7.8 мкК против 7.64 мкК теории; разброс ±0.3 мкК — статистика 400 атомов. Если получаешь вдвое больше или вдвое меньше, проверь «/2» в $\alpha$ и множитель 2 в $\sqrt{2D_p\delta t}$.

Ниже — необязательный скан по отстройке в свободном пространстве против формулы §4.6 (~10 с).

In [ ]:
Ds = np.array([-0.5, -0.8, -1.0, -1.5, -2.0])*Gamma
T_mc = [run_mc(N=300, t_total=4e-3, t_meas=2e-3, free_space=True, T0=10*uK, Delta=D,
               axes=(('xz', 0), ('xz', 90), ('yz', 90)), seed=7)['T'] for D in Ds]
DD = np.linspace(-2.2, -0.3, 100)*Gamma
plt.figure(figsize=(4.5, 3))
plt.plot(DD/Gamma, T_doppler_model(2, DD)/uK, label='теория §4.6'); plt.plot(Ds/Gamma, T_mc, 'o', label='МК')
plt.xlabel('Δ/Γ'); plt.ylabel('T, мкК'); plt.legend(); plt.title('свободное пространство, s=2'); plt.tight_layout(); plt.show()

## 5. Атом в пинцете → Fig. 3(a)

### 5.1 Почему важен угол
Пучок под углом $\theta_k$ к оси пинцета даёт трение по $z$ с весом $\cos^2\theta_k$ и по $x$ (или $y$) с весом $\sin^2\theta_k$.
* При $\theta_k=0$ трение действует только по $z$. Радиальные степени свободы охлаждаются лишь через ангармоническое перемешивание, а их нагрев идёт за счёт изотропной части толчков.
* При $\theta_k=90°$ всё наоборот: не охлаждается ось $z$.
* Оптимум лежит посередине (~50°), где перемешивание успевает выровнять энергию.

В анизотропной ловушке ($w_y=1.1\,w_0$) одна ось в плоскости xz хуже охлаждает $y$. Вторая ось в плоскости yz это исправляет.

### 5.2 Сначала одна траектория ансамбля: как быстро наступает равновесие
Это аналог вставки Fig. 7(b): равновесие за ~1 мс.

In [ ]:
res = run_mc(N=300, t_total=4e-3, t_meas=1e-3, axes=(('xz', 50),), T0=20*uK, seed=3)
plt.figure(figsize=(4.5, 3)); plt.plot(res['t']*1e3, res['T_t'], lw=.7)
plt.axhline(T_doppler_model(2, -Gamma)/uK, ls='--', c='k', label='свободное пр-во')
plt.xlabel('t, мс'); plt.ylabel('T, мкК'); plt.legend(); plt.title('1 ось, 50°, 0.2 мК'); plt.tight_layout(); plt.show()
print(f"T (последняя мс) = {res['T']:.2f} мкК, retention = {res['retention']:.3f}")

**Ожидаемо:** спад от 20 до ~8 мкК за ~1 мс; удержание ≈ 1.

### ✍️ ЗАДАНИЕ 6. Fig. 3(a)
Для каждой из трёх конфигураций и каждого $\theta_k$ из `thetas` запусти `run_mc` с параметрами Fig. 3(a): $s=2$, $\Delta=-\Gamma$, $U_0=0.2$ мК, `t_total=15e-3`, `t_meas=7e-3`, `N=300`. Сохрани $T$ и retention. Нарисуй три кривые $T(\theta_k)$; маркер можно делать полупрозрачным там, где retention < 0.9.

Время счёта: ~25 с на кривую при 8 углах. Для гладкости можно взять больше углов и `N`.

In [ ]:
# ✍️ ЗАДАНИЕ
thetas = np.array([0, 15, 30, 40, 50, 60, 75, 90])
configs = {
    '1-axis':         dict(aniso=0.0, axes=lambda th: (('xz', th),)),
    '1-axis (asym.)': dict(aniso=0.1, axes=lambda th: (('xz', th),)),
    '2-axis (asym.)': dict(aniso=0.1, axes=lambda th: (('xz', th), ('yz', th))),
}
results = {}      # results[name] = (массив T, массив retention)
# ✍️ ВАШ КОД: цикл + график
raise NotImplementedError

### 5.3 Что должно получиться и как это читать
Эталонный прогон (`N=300`, seed=1); разброс ±0.3 мкК:

| θk | 0° | 15° | 30° | 40° | 50° | 60° | 75° | 90° |
|---|---|---|---|---|---|---|---|---|
| 1-axis, мкК | 15.7 | 13.2 | 9.7 | 8.4 | **7.9** | 8.4 | 10.8 | 12.0 |
| 1-axis (asym.) | 15.5 | 13.2 | 10.5 | 9.4 | **9.2** | 9.8 | 12.1 | 14.1 |
| 2-axis (asym.) | 15.5 | 13.8 | 9.8 | 8.2 | **7.6** | 7.9 | 9.7 | 11.2 |
| retention 1-axis | 0.43 | 0.73 | 0.99 | 1.00 | 1.00 | 0.99 | 0.92 | 0.73 |

**Совпадает со статьёй:**
* минимум около 50°;
* порядок кривых в минимуме: 2-axis (asym.) < 1-axis (sym.) < 1-axis (asym.);
* 2 оси в анизотропной ловушке холоднее одной оси даже в симметричной;
* при 0° кривые 1-axis (asym.) и 2-axis совпадают (обе оси вдоль z).

**Отличается:**
1. Абсолютный уровень в минимуме у нас на ~15–20 % ниже (статья: ~8.5 / 10 / 11 мкК). В статье нет нескольких деталей реализации: как именно определена $T$, какова начальная $T$ и длительность, глубина для Fig. 3(a), как разыгрывается $\hat e_p$. Наш минимум для 1 оси почти равен идеальному пределу §4.6, то есть перемешивание у нас эффективнее.
2. На краях (0°, 90°) атомы теряются (retention 0.43–0.73). $T$ по выжившим смещена вниз: горячие атомы уходят. Поэтому у нас на краях 12–16 мкК вместо 17–19 мкК. Это не равновесная температура, а признак «ловушка слишком мелкая для такой геометрии», о чём и говорит статья.

**Эксперименты для понимания:** поставь `aniso=0.2`; замени скалярное трение на векторное; уменьши `dt` до 0.5 мкс и проверь, что $T$ не меняется больше статистики.

## 6. Бонус

### 6.1 Release–recapture (Fig. 3(b), фиолетовая линия)
Протокол: после охлаждения выключаем ловушку на $t_{\rm rel}=20$ мкс; атомы летят свободно ($\vec r\to\vec r+\vec v t_{\rm rel}$). Затем ловушку включаем, и атом пойман, если $\tfrac12mv^2+U(\vec r)<0$. Retention — доля пойманных среди тех, кто был жив до выключения.

### ✍️ ЗАДАНИЕ 7
Напиши `release_recapture(res, U0, t_rel)`, используя `res['r'], res['v'], res['alive'], res['wx'], res['wy']`.

In [ ]:
# ✍️ ЗАДАНИЕ
def release_recapture(res, U0, t_rel):
    # ✍️ ВАШ КОД
    raise NotImplementedError

In [ ]:
# 2 оси (52° в xz, 73° в yz), чередование 2.5 кГц, анизотропия 10 %, s=2, Delta=-Gamma, 0.2 мК
res_rr = run_mc(N=400, t_total=15e-3, t_meas=5e-3, aniso=0.1, axes=(('xz', 52), ('yz', 73)), alt_freq=2.5e3, seed=2)
print(f"T = {res_rr['T']:.2f} мкК,  release-recapture 20 мкс: {release_recapture(res_rr, 0.2*mK*kB, 20e-6):.3f}")

**Ожидаемо:** около 8.1 мкК и retention ≈ 0.94. Фиолетовая линия Fig. 3(b) — около 0.9; эксперимент с чередованием даёт 0.85–0.9.

### 6.2 Потери на испущенный фотон (Fig. 6(a), 7(a))
В мелкой ловушке горячий хвост распределения выбивается через $E\ge0$. Статья показывает, что retention экспоненциально спадает с числом испущенных фотонов: ${\rm ret}=e^{-L\,N_{\rm ph}}$. Поэтому разумная метрика — **потеря на фотон** $L$.

Параметры как у Fig. 7(a): $s=1$, $\Delta=-\sqrt2\,\Gamma/2$, $\gamma=0$ (imaging-magic), чередование 5 кГц, глубина 0.1 мК; три пары углов. Чтобы увидеть потери $L\sim10^{-5}$, нужно $\gtrsim 2\cdot10^4$ фотонов на атом, то есть ~0.15 с модельного времени. Начинаем с $T_0=8$ мкК и отбрасываем первые 2 мс, иначе переходный процесс от горячего старта завышает $L$.

### ✍️ ЗАДАНИЕ 8
Напиши `fit_loss_per_photon(nph, ret)`: МНК для $\ln{\rm ret} = -L\,N_{\rm ph}$ (прямая через ноль), `nph` и `ret` — массивы по времени. Отбрось точки с `nan` и `ret == 0`.

In [ ]:
# ✍️ ЗАДАНИЕ
def fit_loss_per_photon(nph, ret):
    # ✍️ ВАШ КОД
    raise NotImplementedError

In [ ]:
RUN_BONUS = True          # ~2 мин (3 конфигурации по ~35 с)
if RUN_BONUS:
    out = {}
    cfgs = [('52°,52°', (('xz', 52), ('yz', 52))), ('52°,73°', (('xz', 52), ('yz', 73))), ('73°,73°', (('xz', 73), ('yz', 73)))]
    for name, ax_ in cfgs:
        r_ = run_mc(N=300, t_total=0.15, t_meas=1e-3, s=1.0, Delta=-np.sqrt(2)*Gamma/2, U0=0.1*mK*kB,
                    axes=ax_, alt_freq=5e3, T0=8*uK, seed=3)
        i0 = 2000                                   # отбрасываем первые 2 мс (переходный процесс от T0)
        n_, ret_ = r_['nph_t'][i0:] - r_['nph_t'][i0], r_['ret_t'][i0:]/r_['ret_t'][i0]
        L = fit_loss_per_photon(n_, ret_); out[name] = (n_, ret_, L)
        print(f"{name}: T={r_['T']:.2f} мкК, retention={r_['retention']:.3f},  L = {L:.2e} на испущенный фотон")
    plt.figure(figsize=(4.5, 3))
    for name, (n_, ret_, L) in out.items():
        plt.plot(n_, ret_, label=f'{name}, L={L:.1e}')
        plt.plot(n_, np.exp(-L*n_), 'k--', lw=.8)
    plt.xlabel('испущенные фотоны'); plt.ylabel('retention'); plt.legend(); plt.title('0.1 мК, s=1, Δ=-Γ/√2'); plt.tight_layout(); plt.show()

**Ожидаемо (эталонный прогон):**

| углы | T, мкК | L (наш МК) | L (Fig. 7(a), 0.1 мК, на глаз) |
|---|---|---|---|
| 52°, 52° | 6.4 | 1.2·10⁻⁵ | ~3·10⁻⁶ |
| 52°, 73° | 6.9 | 4.9·10⁻⁵ | ~1·10⁻⁵ |
| 73°, 73° | 8.0 | 2.1·10⁻⁴ | ~4·10⁻⁵ |

Порядок конфигураций и масштаб совпадают; наши потери систематически в ~4–5 раз выше. Это ожидаемая чувствительность: потеря — это хвост распределения, $L\propto e^{-U_0/k_BT_{\rm eff}}$ при $U_0/k_BT\approx15$. Различие в 10 % в эффективной глубине или в форме хвоста (детали $\hat e_p$, линейное трение) даёт множитель ~4. Время жизни при этом экспоненциальное, как на Fig. 6(a).

Дальше можно построить $L(U_0)$ для 0.06–0.1 мК (Fig. 7(a)) и $L(f_{\rm alt})$ (Fig. 7(b)): резонанс потерь около $f_{\rm alt}\approx f_r$, где чередование параметрически раскачивает радиальное движение.

## 7. Сводка приближений

| Приближение | Где | Что выброшено |
|---|---|---|
| Двухтоновая схема ≈ двухуровневый атом | §2, (D2) | соседние зеемановские уровни, «чужой» тон, расщепление основного состояния |
| Стационарное внутреннее состояние | §1 | переходные процессы на масштабе $1/\Gamma$; $\omega_r/\Gamma\approx0.2$ |
| Некогерентное сложение пучков | (D2) | интерференция, поляризационные градиенты (Fig. 3(b), зелёная полоса) |
| Линейное по $v$ трение | (D3) | нелинейность силы при $kv\sim\Gamma$ (горячий хвост → важно для потерь) |
| Скалярное трение вдоль $\vec v$ | (D3) | поперечная компонента векторной силы |
| Один гауссов толчок по дипольной диаграмме | (D4) | дискретность фотонов; поглощение вдоль пучка |
| $\nabla U' \approx (1+\gamma\rho_{ee})\nabla U$ | §4.4 | градиент $\rho_{ee}$; дипольная сила света охлаждения |
| Классическое движение | §1 | квантование уровней в ловушке ($k_BT/\hbar\omega_r\approx5$ — не огромный запас) |
| Анизотропия только по $w_x \ne w_y$ | §3 | изменение $z_R$, аберрации |
| Симплектический Эйлер, $\delta t=1$ мкс | §4.5 | ошибки $O((\omega\delta t)^2)$ |
| Потеря при $E\ge0$ | §4.5 | рамановское рассеяние в метастабильные (App. E), фоновый газ |